In [3]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler

from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor

In [4]:
# Load training data
df = pd.read_csv("../train.csv")

In [5]:
# Separate features and target
X = df.drop(columns=["Price_INR_Lakhs","Property_ID"])
y = df["Price_INR_Lakhs"]

In [6]:
# Use the same train-validation split for every model
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]


In [8]:
# Preprocessing for tree-based models
linear_num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("preprocessor",StandardScaler())
])

tree_num_pipeline= Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

linear_preprocessor = ColumnTransformer([
    ("num",linear_num_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])

tree_preprocessor=ColumnTransformer([
    ("num",tree_num_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])



In [9]:
# Define candidate models
models ={
    "Linear Regression":Pipeline([
        ("preprocessor",linear_preprocessor),
        ("regressor",LinearRegression())
    ]),

    "Random Forest":Pipeline([
        ("preprocessor",tree_preprocessor),
        ("regressor",RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boost":Pipeline([
        ("preprocessor",tree_preprocessor),
        ("regressor",HistGradientBoostingRegressor(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=15,
            random_state=42
        ))
    ])
}



In [10]:
# Train and evaluate each model
results=[]

for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred=model.predict(X_val)

    results.append({
        "model":name,
        "Training R2":r2_score(y_train,model.predict(X_train)),
        "Validation R2":r2_score(y_val,y_pred),
        "Validation MAE":mean_absolute_error(y_val,y_pred),
        "Validation RMSE":mean_squared_error(y_val,y_pred) ** 0.5
    })

results_df=pd.DataFrame(results).sort_values("Validation R2",ascending=False)

print("Final Model Comparison")
print("_"*60)
print(results_df.round(4).to_string(index=False))




Final Model Comparison
____________________________________________________________
            model  Training R2  Validation R2  Validation MAE  Validation RMSE
   Gradient Boost       0.8024         0.7896         32.7198          68.2540
    Random Forest       0.8824         0.7847         33.1000          69.0478
Linear Regression       0.6710         0.6657         47.9403          86.0255


In [11]:
print(results)

[{'model': 'Linear Regression', 'Training R2': 0.6710086784592575, 'Validation R2': 0.6657463774890661, 'Validation MAE': 47.9402540507363, 'Validation RMSE': 86.02550215479064}, {'model': 'Random Forest', 'Training R2': 0.8824457006341203, 'Validation R2': 0.7846617241547978, 'Validation MAE': 33.10003978309968, 'Validation RMSE': 69.04778809764423}, {'model': 'Gradient Boost', 'Training R2': 0.8023929660273045, 'Validation R2': 0.7895844146648253, 'Validation MAE': 32.7197536300375, 'Validation RMSE': 68.25399987713476}]
